# Simulasi Model Ising Termodifikasi untuk Opini Publik
### Studi Kasus: Social Listening Multi-Platform — Isu Pemerintahan Prabowo Subianto

Notebook ini mengimplementasikan simulasi Monte Carlo (algoritma Metropolis-Hastings) dari Hamiltonian termodifikasi:

$$H = -\sum_{i<j} J_0\, A_{ij}\, \delta(\text{Media}_i,\text{Media}_j)\, \delta(\text{Category}_i,\text{Category}_j)\, e^{-|\Delta t_{ij}|/\tau}\; s_i s_j \;-\; \sum_i \big[h_{\text{media}}(\text{Media}_i) + h_{\text{event}}(t_i)\big]\, s_i$$

dengan $s_i = w(\text{Majas}_i) \in [-1, +1]$.

**Struktur notebook:**
1. Setup & konfigurasi
2. Pemuatan & pembersihan data
3. Pemetaan spin $s_i = w(\text{Majas}_i)$
4. Konstruksi graf interaksi $A_{ij}$ dari kolom `Type` / `Link`
5. Konstruksi $J_{ij}$ (homofili platform, homofili isu, peluruhan temporal)
6. Konstruksi medan eksternal $h_i(t)$
7. Simulasi Metropolis-Hastings
8. Estimasi parameter (inverse Ising / moment matching)
9. Visualisasi & interpretasi


## 1. Setup & Konfigurasi

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from dataclasses import dataclass, field
from datetime import datetime
import re
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)

# Path dataset
DATA_PATH = "dataset.csv"

In [ ]:
@dataclass
class Config:
    J0: float = 1.0
    tau: float = 3.0
    beta: float = 1.0
    n_sweeps: int = 2000
    burn_in: int = 500
    thin: int = 5
    max_nodes: int = 3000 # untuk simulasi FINAL (setelah parameter didapat)
    step_size: float = 0.2

    # khusus tahap kalibrasi (Tahap 8), sengaja jauh lebih kecil
    calib_max_nodes: int = 300  # subsample kecil hanya untuk fitting J0/tau/beta
    calib_n_sweeps: int = 80
    calib_maxiter: int = 15

CFG = Config()

## 2. Pemuatan & Pembersihan Data

Catatan dataset: terdapat masalah mojibake (double-encoding UTF-8), duplikasi konten, dan sel multi-baris. Tahap ini melakukan pembersihan minimal yang relevan untuk simulasi (bukan pembersihan teks lengkap).

In [ ]:
def fix_mojibake(text: str) -> str:
    """Perbaikan heuristik untuk double-encoding UTF-8 (mis. 'Ã°ÂŸÂ¤Â£').
    Mencoba encode kembali sebagai latin1 lalu decode sebagai utf-8."""
    if not isinstance(text, str):
        return text
    try:
        return text.encode("latin1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return text

def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8")

    # Kolom ke-11 tanpa header -> beri nama eksplisit
    if df.columns[-1].startswith("Unnamed"):
        df = df.rename(columns={df.columns[-1]: "Creator"})

    # Perbaikan mojibake pada kolom teks utama
    for col in ["Mentions"]:
        if col in df.columns:
            df[col] = df[col].apply(fix_mojibake)

    # Parsing tanggal
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Buang baris tanpa tanggal / tanpa Majas (tidak dapat dipetakan ke spin)
    df = df.dropna(subset=["Date", "Majas"]).reset_index(drop=True)

    # Deduplikasi berbasis (Media, Mentions) untuk mengurangi inflasi
    # korelasi akibat retweet berantai / cross-post
    df = df.drop_duplicates(subset=["Media", "Mentions"]).reset_index(drop=True)

    # Normalisasi kategori "-" menjadi NaN eksplisit
    for col in ["Category Group", "Category"]:
        if col in df.columns:
            df[col] = df[col].replace("-", np.nan)

    return df

df = load_dataset(DATA_PATH)
print(f"Jumlah baris setelah pembersihan: {len(df)}")
df.head()

In [ ]:
# Subsample agar ukuran graf & simulasi tetap tractable (opsional, lihat CFG.max_nodes)
if len(df) > CFG.max_nodes:
    df = df.sample(n=CFG.max_nodes, random_state=42).reset_index(drop=True)

df["node_id"] = df.index
N = len(df)
print(f"Jumlah node (unit opini) yang disimulasikan: {N}")

## 3. Pemetaan Spin $s_i = w(\text{Majas}_i)$

Karena seluruh baris berlabel `Sentiment = Negative`, variasi dimodelkan melalui intensitas gaya bahasa (`Majas`), bukan polaritas sentimen.

In [ ]:
# Peta bobot majas -> spin kontinu s_i in [-1, +1]
# Nilai bersifat indikatif dan dapat dikalibrasi ulang lewat expert scoring / validasi antar-anotator.
MAJAS_WEIGHTS = {
    "LITERAL":    -1.0,
    "PENEGASAN":  -1.0,
    "REPETISI":   -0.5,
    "METAFORA":   -0.25,
    "SIMILE":     -0.25,
    "SINDIRAN":    0.0,
    "IRONI":       0.0,
    "RETORIS":     0.0,
    "KRITIK":      0.5,
    "SINISME":     0.5,
    "SARKASME":    1.0,
    "SATIRE":      1.0,
    "HIPERBOLA":   1.0,
}

def majas_to_spin(label: str) -> float:
    if not isinstance(label, str):
        return 0.0
    key = label.strip().upper()
    return MAJAS_WEIGHTS.get(key, 0.0)  # default netral bila label tidak dikenali

df["s"] = df["Majas"].apply(majas_to_spin)

print(df["s"].describe())
df["s"].value_counts().sort_index()

## 4. Konstruksi Graf Interaksi $A_{ij}$

Edge didefinisikan dari struktur relasi yang tersirat pada kolom `Type` dan `Link`:
- Twitter/X: `rt` / `reply` -> edge ke node sumber (didekati lewat kesamaan target `Link`)
- Facebook/Threads/Instagram: `*-comment` -> edge komentar terhadap unggahan yang sama
- Node dengan `Link` unggahan induk yang sama dikelompokkan sebagai satu *thread*,
  lalu dihubungkan sebagai graf bintang (komentar <-> unggahan, dan antar-komentar
  pada thread yang sama diberi edge lemah).

Jika data reply/parent-post eksplisit tidak tersedia, digunakan pendekatan **grouping
per Link dasar** (menghilangkan parameter query seperti `?comment_id=`) sebagai proxy
struktur thread.

In [ ]:
def base_link(url: str) -> str:
    """Menyederhanakan Link menjadi identitas thread/unggahan induk,
    membuang parameter query seperti '?comment_id=' pada Facebook."""
    if not isinstance(url, str):
        return "unknown"
    return url.split("?")[0].rstrip("/")


def is_comment_type(type_val: str) -> bool:
    if not isinstance(type_val, str):
        return False
    t = type_val.lower()
    return t.endswith("-comment") or t in {"reply", "rt", "mention"}


df["thread_id"] = df["Link"].apply(base_link)

def build_interaction_graph(df: pd.DataFrame) -> nx.Graph:
    G = nx.Graph()
    G.add_nodes_from(df["node_id"])

    # Edge dalam satu thread (unggahan + komentar-komentarnya)
    for thread_id, group in df.groupby("thread_id"):
        ids = group["node_id"].tolist()
        if len(ids) < 2:
            continue
        # node non-komentar (post/video/photo) dianggap "pusat" thread bila ada
        types = group["Type"].fillna("")
        centers = group.loc[~types.apply(is_comment_type), "node_id"].tolist()
        comments = group.loc[types.apply(is_comment_type), "node_id"].tolist()

        if centers:
            # graf bintang: setiap komentar terhubung ke pusat unggahan
            for c in centers:
                for k in comments:
                    G.add_edge(c, k, relation="comment_on_post")
            # antar pusat (jika >1) dihubungkan juga
            for a, b in zip(centers, centers[1:]):
                G.add_edge(a, b, relation="same_thread")
        else:
            # tidak ada pusat eksplisit -> hubungkan berurutan sebagai rantai balasan
            for a, b in zip(ids, ids[1:]):
                G.add_edge(a, b, relation="sequential_in_thread")

    return G


G = build_interaction_graph(df)
print(f"Jumlah edge pada graf interaksi: {G.number_of_edges()}")
print(f"Jumlah node terisolasi: {sum(1 for n in G.nodes if G.degree(n) == 0)}")

## 5. Konstruksi $J_{ij}$: Homofili Platform, Homofili Isu, Peluruhan Temporal

$$J_{ij} = J_0 \cdot A_{ij} \cdot \delta(\text{Media}_i,\text{Media}_j) \cdot \delta(\text{Category}_i,\text{Category}_j) \cdot e^{-|\Delta t_{ij}|/\tau}$$

$J_{ij}$ hanya dihitung pada edge yang sudah ada di $A_{ij}$ (graf sparse), bukan
matriks lengkap $N \times N$, agar tetap efisien.

In [ ]:
def compute_J(G: nx.Graph, df: pd.DataFrame, J0: float, tau: float) -> nx.Graph:
    dates = df["Date"]
    media = df["Media"]
    category = df["Category"]

    for u, v in G.edges():
        delta_t_days = abs((dates.iloc[u] - dates.iloc[v]).total_seconds()) / 86400.0
        same_media = 1.0 if media.iloc[u] == media.iloc[v] else 0.0
        same_category = 1.0 if (
            pd.notna(category.iloc[u]) and pd.notna(category.iloc[v])
            and category.iloc[u] == category.iloc[v]
        ) else 0.0

        # delta(Category) longgar: bila salah satu kategori tidak diketahui ("-"), gunakan bobot netral 0.5
        # alih-alih 0, agar edge tidak mati total hanya karena kategorisasi isu belum lengkap
        if pd.isna(category.iloc[u]) or pd.isna(category.iloc[v]):
            cat_factor = 0.5
        else:
            cat_factor = same_category

        kappa = np.exp(-delta_t_days / tau)

        J_uv = J0 * same_media * cat_factor * kappa
        G[u][v]["J"] = J_uv

    return G

G = compute_J(G, df, CFG.J0, CFG.tau)

J_values = np.array([d["J"] for _, _, d in G.edges(data=True)])
print(f"J_ij  -> mean={J_values.mean():.4f}, std={J_values.std():.4f}, "
      f"min={J_values.min():.4f}, max={J_values.max():.4f}")

## 6. Konstruksi Medan Eksternal $h_i(t)$

$$h_i(t) = h_{\text{media}}(\text{Media}_i) + h_{\text{event}}(t_i)$$

- $h_{\text{media}}$: rata-rata intensitas spin dasar tiap platform (bias struktural
  platform berita vs. platform personal).
- $h_{\text{event}}(t)$: KDE 1-D pada sumbu waktu dari volume postingan per hari,
  dinormalisasi, sebagai proxy "guncangan peristiwa" (lonjakan volume opini pada
  tanggal tertentu, mis. saat berita pelemahan rupiah/IHSG mencuat).

In [ ]:
def compute_h_media(df: pd.DataFrame) -> dict:
    """h_media: rata-rata spin per platform, dicentering terhadap rata-rata
    global sehingga berperan sebagai bias relatif, bukan level absolut."""
    means = df.groupby("Media")["s"].mean()
    return (means - means.mean()).to_dict()


def compute_h_event(df: pd.DataFrame, bandwidth_days: float = 1.5) -> pd.Series:
    """Proxy guncangan peristiwa: volume postingan per hari, dihaluskan dengan
    kernel Gaussian, lalu dinormalisasi ke rentang [0, 1]."""
    daily_counts = df.groupby(df["Date"].dt.date).size()
    idx = pd.date_range(daily_counts.index.min(), daily_counts.index.max(), freq="D")
    daily_counts = daily_counts.reindex(idx.date, fill_value=0)

    days = np.arange(len(daily_counts))
    smoothed = np.zeros_like(days, dtype=float)
    for i in days:
        weights = np.exp(-0.5 * ((days - i) / bandwidth_days) ** 2)
        smoothed[i] = np.sum(weights * daily_counts.values) / weights.sum()

    smoothed = (smoothed - smoothed.min()) / (smoothed.max() - smoothed.min() + 1e-9)
    return pd.Series(smoothed, index=pd.to_datetime(idx))

h_media_map = compute_h_media(df)
h_event_series = compute_h_event(df)

def h_field(row, h_event_scale: float = 1.0) -> float:
    day = pd.Timestamp(row["Date"].date())
    h_ev = h_event_series.get(day, 0.0)
    return h_media_map.get(row["Media"], 0.0) + h_event_scale * h_ev

df["h_i"] = df.apply(h_field, axis=1)
print(df.groupby("Media")["h_i"].mean())

## 7. Simulasi Metropolis-Hastings

Karena $s_i$ didefinisikan kontinu pada $[-1, +1]$ (bukan biner $\{-1,+1\}$ murni),
digunakan varian **Ising kontinu (soft-spin)**: proposal berupa perturbasi kecil
$s_i' = \text{clip}(s_i + \epsilon, -1, 1)$, diterima/ditolak dengan kriteria Metropolis
standar terhadap $\Delta H$.

In [44]:
def build_neighbor_arrays(G: nx.Graph):
    """Konversi graf ke daftar tetangga + bobot J dalam bentuk numpy array
    per node, agar loop simulasi tidak bolak-balik lookup ke objek networkx."""
    n = G.number_of_nodes()
    neighbors = [np.array([], dtype=int) for _ in range(n)]
    weights = [np.array([], dtype=float) for _ in range(n)]
    adj = {i: ([], []) for i in range(n)}
    for u, v, d in G.edges(data=True):
        adj[u][0].append(v); adj[u][1].append(d["J"])
        adj[v][0].append(u); adj[v][1].append(d["J"])
    for i in range(n):
        neighbors[i] = np.array(adj[i][0], dtype=int)
        weights[i] = np.array(adj[i][1], dtype=float)
    return neighbors, weights


def local_energy_fast(s, node, neighbors, weights, h):
    nbrs = neighbors[node]
    if len(nbrs) == 0:
        interaction = 0.0
    else:
        interaction = np.dot(weights[node], s[nbrs]) * s[node]
    return -interaction - h[node] * s[node]


def metropolis_sweep_fast(s, neighbors, weights, h, beta, step_size, rng):
    order = rng.permutation(len(s))
    for node in order:
        E_old = local_energy_fast(s, node, neighbors, weights, h)
        s_old = s[node]
        s[node] = np.clip(s_old + rng.uniform(-step_size, step_size), -1.0, 1.0)
        E_new = local_energy_fast(s, node, neighbors, weights, h)
        dE = E_new - E_old
        if dE > 0 and rng.random() >= np.exp(-beta * dE):
            s[node] = s_old
    return s


def run_simulation(df, G, cfg: Config, s_init=None, verbose=False):
    rng = np.random.default_rng(42)
    h = df["h_i"].values
    s = s_init.copy() if s_init is not None else df["s"].values.copy()
    neighbors, weights = build_neighbor_arrays(G)

    samples, magnetization_trace = [], []
    for sweep in range(cfg.n_sweeps):
        s = metropolis_sweep_fast(s, neighbors, weights, h, cfg.beta, cfg.step_size, rng)
        magnetization_trace.append(s.mean())
        if sweep >= cfg.burn_in and (sweep - cfg.burn_in) % cfg.thin == 0:
            samples.append(s.copy())
        if verbose and sweep % max(1, cfg.n_sweeps // 5) == 0:
            print(f"  sweep {sweep}/{cfg.n_sweeps}  <s>={s.mean():.4f}")

    return {"final_state": s, "samples": np.array(samples),
            "magnetization_trace": np.array(magnetization_trace)}

result = run_simulation(df, G, CFG)

## 8. Estimasi Parameter (Inverse Ising / Moment Matching)

Kerangka teoretis: parameter $(J_0, \tau, \beta)$ diestimasi dengan mencocokkan momen model terhadap momen empiris:

$$\langle s_i \rangle_{\text{model}} = \langle s_i \rangle_{\text{data}}, \qquad
\langle s_i s_j \rangle_{\text{model}} = \langle s_i s_j \rangle_{\text{data}}$$

Pendekatan yang digunakan: *simulated moment matching* — mencari $(J_0, \tau, \beta)$
yang meminimalkan selisih kuadrat antara momen simulasi dan momen data empiris pada
edge graf $G$.

In [ ]:
def make_calibration_subset(df, G, n_calib, seed=42):
    """Ambil subgraf kecil (bukan seluruh N) khusus untuk fitting parameter."""
    rng = np.random.default_rng(seed)
    all_nodes = list(G.nodes())

    if len(all_nodes) <= n_calib:
        sub_nodes = all_nodes
    else:
        sub_nodes = rng.choice(all_nodes, size=n_calib, replace=False)

    G_sub = G.subgraph(sub_nodes).copy()
    mapping = {old: new for new, old in enumerate(sorted(G_sub.nodes()))}
    G_sub = nx.relabel_nodes(G_sub, mapping)
    
    # Cek apakah node_id ada di dalam kolom atau sudah menjadi index
    if "node_id" in df.columns:
        df_temp = df.set_index("node_id")
    else:
        df_temp = df
        
    df_sub = df_temp.loc[sorted(mapping.keys())].reset_index(drop=True)
    df_sub["node_id"] = df_sub.index
    return df_sub, G_sub

def empirical_moments(df, G):
    """Hitung momen dari data empiris untuk dicocokkan."""
    s = df["s"].values
    pair_products = [s[u] * s[v] for u, v in G.edges()]
    return {"mean_s": s.mean(), "mean_ss": np.mean(pair_products) if pair_products else 0.0}

def model_moments_from_samples(samples, G):
    """Hitung momen dari hasil sampling simulasi model."""
    edge_list = list(G.edges())
    ss = [np.mean([sample[u] * sample[v] for u, v in edge_list]) for sample in samples]
    return {"mean_s": samples.mean(), "mean_ss": np.mean(ss)}

# Global log untuk memantau proses optimasi
_eval_log = {"count": 0, "start": None}

def moment_matching_loss(params, df_c, G_c, target, cfg: Config):
    import time
    if _eval_log["start"] is None:
        _eval_log["start"] = time.time()

    J0, tau, beta = params

    # Beri penalti besar jika parameter bernilai negatif atau nol
    if J0 <= 0 or tau <= 0 or beta <= 0:
        return 1e6

    t0 = time.time()

    # Hitung matriks J pada subgraf kalibrasi
    G_fit = compute_J(G_c, df_c, J0, tau)

    # Konfigurasi parameter khusus untuk kalibrasi (menggunakan iterasi lebih sedikit)
    cfg_fit = Config(
        J0=J0, 
        tau=tau, 
        beta=beta,
        n_sweeps=cfg.calib_n_sweeps,
        burn_in=cfg.calib_n_sweeps // 3, 
        thin=5
    )

    # Jalankan simulasi pada subsample
    result = run_simulation(df_c, G_fit, cfg_fit)
    model_m = model_moments_from_samples(result["samples"], G_fit)
    
    # Hitung selisih kuadrat momen
    loss = (model_m["mean_s"] - target["mean_s"]) ** 2 + (model_m["mean_ss"] - target["mean_ss"]) ** 2

    _eval_log["count"] += 1
    dt = time.time() - t0
    elapsed = time.time() - _eval_log["start"]
    avg = elapsed / _eval_log["count"]

    print(f"[eval {_eval_log['count']:03d}] J0={J0:.3f} tau={tau:.3f} beta={beta:.3f} "
          f"loss={loss:.5f}  ({dt:.1f}s, rata-rata {avg:.1f}s/eval)")

    return loss

# Jalankan kalibrasi pada subsample kecil
# Ukuran graf dibatasi oleh variabel calib_max_nodes
df_calib, G_calib = make_calibration_subset(df, G, CFG.calib_max_nodes)
G_calib = compute_J(G_calib, df_calib, CFG.J0, CFG.tau)
target_moments = empirical_moments(df_calib, G_calib)

print("Memulai proses estimasi parameter...")
print(f"Target momen empiris: mean_s={target_moments['mean_s']:.4f}, mean_ss={target_moments['mean_ss']:.4f}\n")

opt_result = minimize(
    moment_matching_loss,
    x0=[CFG.J0, CFG.tau, CFG.beta],
    args=(df_calib, G_calib, target_moments, CFG),
    method="Nelder-Mead",
    options={"maxiter": CFG.calib_maxiter, "xatol": 1e-2, "fatol": 1e-3},
)

J0_hat, tau_hat, beta_hat = opt_result.x
print(f"\nEstimasi akhir yang diperoleh: J0={J0_hat:.3f}, tau={tau_hat:.3f}, beta={beta_hat:.3f}")

## 9. Visualisasi & Interpretasi

Setelah simulasi dijalankan (`result = run_simulation(df, G, CFG)`), sel-sel berikut
menghasilkan visualisasi untuk interpretasi:
1. Jejak magnetisasi $\langle s \rangle$ per sweep (konvergensi / burn-in)
2. Distribusi spin akhir dibanding distribusi empiris (`Majas`)
3. Peta panas korelasi $J_{ij}$ vs platform
4. Kurva $h_{\text{event}}(t)$ terhadap linimasa tanggal (deteksi guncangan peristiwa)

In [ ]:
def plot_magnetization_trace(result: dict, cfg: Config):
    plt.figure(figsize=(8, 3))
    plt.plot(result["magnetization_trace"], lw=0.8)
    plt.axvline(cfg.burn_in, color="red", linestyle="--", label="akhir burn-in")
    plt.xlabel("Sweep ke-")
    plt.ylabel(r"Magnetisasi $\langle s \rangle$")
    plt.title("Konvergensi Magnetisasi terhadap Sweep Monte Carlo")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_spin_distribution(df: pd.DataFrame, final_state: np.ndarray):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].hist(df["s"], bins=20, color="steelblue")
    axes[0].set_title("Distribusi spin empiris (dari Majas)")
    axes[0].set_xlabel("s_i")

    axes[1].hist(final_state, bins=20, color="indianred")
    axes[1].set_title("Distribusi spin hasil simulasi (state akhir)")
    axes[1].set_xlabel("s_i")
    plt.tight_layout()
    plt.show()

def plot_h_event(h_event_series: pd.Series):
    plt.figure(figsize=(9, 3))
    plt.plot(h_event_series.index, h_event_series.values)
    plt.xlabel("Tanggal")
    plt.ylabel(r"$h_{event}(t)$ (ternormalisasi)")
    plt.title("Proxy Guncangan Peristiwa dari Volume Postingan Harian")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

def plot_J_by_media(G: nx.Graph, df: pd.DataFrame):
    media = df["Media"]
    records = []
    for u, v, d in G.edges(data=True):
        records.append({"media_pair": f"{media.iloc[u]}-{media.iloc[v]}", "J": d["J"]})
    j_df = pd.DataFrame(records)
    j_df.boxplot(column="J", by="media_pair", rot=90, figsize=(9, 4))
    plt.title("Distribusi $J_{ij}$ per Pasangan Platform")
    plt.suptitle("")
    plt.ylabel(r"$J_{ij}$")
    plt.tight_layout()
    plt.show()

plot_magnetization_trace(result, CFG)
plot_spin_distribution(df, result["final_state"])
plot_h_event(h_event_series)
plot_J_by_media(G, df)

## 10. Interpretasi Akhir
- $\hat{J}_0$ besar dan signifikan → indikasi *eskalasi retoris* (sarcasm cascades) antar warganet yang saling terhubung dalam satu thread/platform/isu.
- $\hat{\tau}$ kecil → pengaruh antar opini cepat meluruh terhadap waktu (reaksi jangka pendek terhadap peristiwa).
- Puncak $h_{\text{event}}(t)$ dapat disandingkan dengan linimasa berita riil (mis. tanggal pengumuman kebijakan) untuk memvalidasi apakah lonjakan intensitas kritik memang berkorelasi dengan peristiwa kebijakan tertentu.
- Perbandingan $\hat{h}_{\text{media}}$ antar platform mengonfirmasi/menolak pola pada dataset (YouTube/TikTok lebih “dingin” karena didominasi akun berita, Threads/Twitter lebih “panas” karena didominasi komentar personal).
